In [1]:
#all imports
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

MODEL_PATH = "../models/cyberguard_calibrated_svm.pkl"
TFIDF_PATH = "../models/cyberguard_tfidf.pkl"

calibrated_svm = joblib.load(MODEL_PATH)
tfidf = joblib.load(TFIDF_PATH)

print("Calibrated SVM loaded")
print("TF-IDF loaded")

Calibrated SVM loaded
TF-IDF loaded


In [2]:
#creating the risk scoring function
def calculate_text_risk(suspicious_probability):
    
    p = suspicious_probability * 100
    
    if p < 10:
        risk_level = "Safe"
    elif p < 30:
        risk_level = "Low"
    elif p < 60:
        risk_level = "Medium"
    elif p < 85:
        risk_level = "High"
    else:
        risk_level = "Critical"
    
    return {
        "suspicious_probability": suspicious_probability,
        "risk_score": p,
        "risk_level": risk_level
    }

In [3]:
#creating the complete text analysis function
def analyze_message(text):
    
    text = str(text)
    
    # Transform text using trained TF-IDF
    X = tfidf.transform([text])
    
    # Prediction
    prediction = calibrated_svm.predict(X)[0]
    
    # Probability
    probability = calibrated_svm.predict_proba(X)[0][1]
    
    # Risk
    risk = calculate_text_risk(probability)
    
    result = {
        "prediction": "Suspicious" if prediction == 1 else "Legitimate",
        "suspicious_probability": probability,
        "risk_score": risk["risk_score"],
        "risk_level": risk["risk_level"]
    }
    
    return result

In [4]:
#a manual test
test_messages = [
    "Hi John, please find the meeting report attached. Regards, Team.",
    
    "URGENT! Your account has been suspended. Click here immediately "
    "to verify your password and avoid permanent account closure.",
    
    "Congratulations! You have won a prize. Click the link below "
    "to claim your reward.",
    
    "Please find the project report attached. We will discuss it "
    "during tomorrow's meeting."
]

for message in test_messages:
    
    result = analyze_message(message)
    
    print("=" * 70)
    print("MESSAGE:")
    print(message)
    print()
    print("Prediction:", result["prediction"])
    print("Suspicious Probability:",
          round(result["suspicious_probability"], 4))
    print("Risk Score:",
          round(result["risk_score"], 2))
    print("Risk Level:",
          result["risk_level"])

MESSAGE:
Hi John, please find the meeting report attached. Regards, Team.

Prediction: Legitimate
Suspicious Probability: 0.0
Risk Score: 0.0
Risk Level: Safe
MESSAGE:
URGENT! Your account has been suspended. Click here immediately to verify your password and avoid permanent account closure.

Prediction: Suspicious
Suspicious Probability: 0.9998
Risk Score: 99.98
Risk Level: Critical
MESSAGE:
Congratulations! You have won a prize. Click the link below to claim your reward.

Prediction: Suspicious
Suspicious Probability: 0.9939
Risk Score: 99.39
Risk Level: Critical
MESSAGE:
Please find the project report attached. We will discuss it during tomorrow's meeting.

Prediction: Legitimate
Suspicious Probability: 0.0
Risk Score: 0.0
Risk Level: Safe


In [5]:
#adding evidence/explanation
import re

def extract_text_indicators(text):
    
    text_lower = text.lower()
    
    indicators = []
    
    # Urgency
    urgency_words = [
        "urgent",
        "immediately",
        "as soon as possible",
        "action required",
        "act now",
        "final warning",
        "suspended",
        "expire"
    ]
    
    for word in urgency_words:
        if word in text_lower:
            indicators.append(
                f"Urgency indicator detected: '{word}'"
            )
    
    # Credential-related requests
    credential_words = [
        "password",
        "username",
        "login",
        "verify your account",
        "credential",
        "otp",
        "one time password",
        "security code"
    ]
    
    for word in credential_words:
        if word in text_lower:
            indicators.append(
                f"Credential-related indicator detected: '{word}'"
            )
    
    # Suspicious URLs
    urls = re.findall(
        r'https?://[^\s]+|www\.[^\s]+',
        text
    )
    
    if urls:
        indicators.append(
            f"{len(urls)} URL(s) detected in the message"
        )
    
    # Reward/scam language
    reward_words = [
        "winner",
        "won",
        "prize",
        "reward",
        "claim",
        "lottery"
    ]
    
    for word in reward_words:
        if word in text_lower:
            indicators.append(
                f"Potential fraud/reward indicator detected: '{word}'"
            )
    
    return indicators

In [6]:
#combining ML and evidence
def analyze_message_full(text):
    
    result = analyze_message(text)
    
    indicators = extract_text_indicators(text)
    
    result["indicators"] = indicators
    
    if result["risk_level"] in ["High", "Critical"]:
        result["recommended_action"] = (
            "Do not click links or provide credentials. "
            "Verify the sender through an independent channel."
        )
    
    elif result["risk_level"] == "Medium":
        result["recommended_action"] = (
            "Exercise caution and verify the sender and links "
            "before taking any action."
        )
    
    elif result["risk_level"] == "Low":
        result["recommended_action"] = (
            "Message appears relatively safe, but normal caution "
            "is recommended."
        )
    
    else:
        result["recommended_action"] = (
            "No immediate threat detected."
        )
    
    return result

In [7]:
#testing the complete cyberguard result
message = """
URGENT! Your bank account has been suspended.
Verify your password immediately by visiting
http://example-login.com/verify.
Failure to act will result in permanent account closure.
"""

result = analyze_message_full(message)

print("CYBERGUARD ANALYSIS")
print("=" * 60)

for key, value in result.items():
    print(f"{key}: {value}")

CYBERGUARD ANALYSIS
prediction: Suspicious
suspicious_probability: 0.9999258243802792
risk_score: 99.99258243802792
risk_level: Critical
indicators: ["Urgency indicator detected: 'urgent'", "Urgency indicator detected: 'immediately'", "Urgency indicator detected: 'suspended'", "Credential-related indicator detected: 'password'", "Credential-related indicator detected: 'login'", '1 URL(s) detected in the message']
recommended_action: Do not click links or provide credentials. Verify the sender through an independent channel.
